$$
判断矩阵 / 正互反矩阵 (不是一致矩阵) \\
\begin{array}{|c|c|c|c|c|}
\hline
 & 粉丝数 & 颜值 & 作品质量 & 作品数量 \\
\hline
粉丝数 & 1 & 2 & 3 & 5 \\
\hline
颜值 & \frac{1}{2} & 1 & \frac{1}{2} & 2 \\
\hline
作品质量 & \frac{1}{3} & 2 & 1 & \frac{1}{2} \\
\hline
作品数量 & \frac{1}{5} & \frac{1}{2} & 2 & 1 \\
\hline
\end{array}
\newline
注意：这里的所有判断标准出自主观，即判断矩阵的元素都是主观判断的结果。\\
$$

In [2]:
import numpy as np

m = np.array([[1, 2, 3, 5], [1/2, 1, 1/2, 2], [1/3, 2, 1, 1/2], [1/5, 1/2, 2, 1]])


$$
一致性检验 \\
\text{Consistency Index (CI) = }\frac{\lambda_{max} - n}{n - 1} \\
\text{CI是一致性指标} \\
\text{Random Index (RI) 是随机一致性指标} \\
\begin{array}{|c|c|c|c|c|c|c|c|c|c|c|c|}
\hline
n & 1 & 2 & 3 & 4 & 5 & 6 & 7 & 8 & 9 & 10 & 11 \\
\hline
RI & 0 & 0 & 0.52 & 0.89 & 1.12 & 1.26 & 1.36 & 1.41 & 1.46 & 1.49 & 1.52 \\
\hline
\end{array}\\
\text{Consistency Ratio (CR) = }\frac{CI}{RI} \\
\text{CR是一致性比例, 当CR小于0.1时, 认为矩阵是一致的, 越小越好} \\
$$

In [3]:
eigvals, eigvecs = np.linalg.eig(m)
eigvals

array([ 4.46724532+0.j        , -0.08901902+0.j        ,
       -0.18911315+1.42051623j, -0.18911315-1.42051623j])

In [4]:
def consistency_analysis(eigvals):
    # 去掉虚数
    eigvals = eigvals.real
    CI = (eigvals.max() - eigvals.size) / (eigvals.size - 1)
    RI = {1: 0, 2: 0, 3: 0.52, 4: 0.89, 5: 1.12, 6: 1.26, 7: 1.36, 8: 1.41, 9: 1.46, 10: 1.49, 11: 1.52}
    CR = CI / RI[eigvals.size]
    print(f'一致性指标 CI = {CI:.4f}\n随机一致性指标 RI = {RI[eigvals.size]:.4f}\n一致性比率 CR = {CR:.4f}')
    if CR == 0.1:
        print('判断矩阵是一致矩阵')
    elif CR < 0.1:
        print('判断矩阵是一致的')
    else:
        print('判断矩阵不是一致的, 建议重新构造判断矩阵')

consistency_analysis(eigvals)


一致性指标 CI = 0.1557
随机一致性指标 RI = 0.8900
一致性比率 CR = 0.1750
判断矩阵不是一致的, 建议重新构造判断矩阵


In [5]:
m = np.array([[1, 2, 3, 5], [1/2, 1, 1/2, 2], [1/3, 2, 1, 2], [1/5, 1/2, 1/2, 1]])
eigvals, eigvecs = np.linalg.eig(m)
consistency_analysis(eigvals)

一致性指标 CI = 0.0376
随机一致性指标 RI = 0.8900
一致性比率 CR = 0.0423
判断矩阵是一致的


$$
下面有三种权重计算方式 \\
$$

$$
\text{1. } 列算术平均 \\
求得每列的总和，再将每列的元素除以总和，再计算每列的总和，最后除以阶数 \\
\textbf{w}_i = \frac{1}{n} \sum_{j=1}^n \frac{a_{ij}}{\sum_{k=1}^n a_{kj}}
$$

In [6]:
col_sums = np.sum(m, axis=0)
weight_arithmic_mean_m = m / col_sums
weight_arithmic_mean_m

array([[0.49180328, 0.36363636, 0.6       , 0.5       ],
       [0.24590164, 0.18181818, 0.1       , 0.2       ],
       [0.16393443, 0.36363636, 0.2       , 0.2       ],
       [0.09836066, 0.09090909, 0.1       , 0.1       ]])

In [7]:
weight_arithmic_mean_m.sum(axis=1)

array([1.95543964, 0.72771982, 0.92757079, 0.38926975])

In [8]:
weight_arithmic_mean_m.sum(axis=1) / weight_arithmic_mean_m.shape[1]

array([0.48885991, 0.18192996, 0.2318927 , 0.09731744])

$$
\text{2. } 列几何平均 \\
按行相乘，再对每个分量开n次方，再对向量归一化 \\
\textbf{w}_i = \frac{\left( \prod_{j=1}^n m_{ij} \right)^{\frac{1}{n}}}{\sum_{k=1}^n \left( \prod_{j=1}^n m_{kj} \right)^{\frac{1}{n}}}
$$

In [9]:
row_product = np.prod(m, axis=1)
weight_geometric_mean_m = row_product**(1/m.shape[1])
weight_geometric_mean_m

array([2.34034732, 0.84089642, 1.07456993, 0.4728708 ])

In [10]:
weight_geometric_mean_m / weight_geometric_mean_m.sum(axis=0)

array([0.49492567, 0.17782883, 0.22724501, 0.1000005 ])

$$
\text{3. } 特征向量 \\
\text{将求得的特征值中最大的代入到 }\textbf{Ax} = \lambda_{max} I \textbf{x} \text{中, 即可得到对应的特征向量} \textbf{x}_{max} \\
\text{接着再进行归一化, 即可得到最终的权重} \\
\textbf{w}_i = \frac{\textbf{x}_i}{\sum_{j=1}^n \textbf{x}_j}
$$

In [11]:
eig_values, eig_vectors = np.linalg.eig(m)
eig_values.real.argmax()

0

In [14]:
final_eig_vector = eig_vectors[:, eig_values.real.argmax()].real
final_eig_vector

array([0.84869741, 0.30763892, 0.39621458, 0.16758584])

In [15]:
final_eig_vector / final_eig_vector.sum()

array([0.4933895 , 0.17884562, 0.230339  , 0.09742588])